# BR01A — Batch-matched RAD-DINO recheck

Use after a borderline single-image verification failure. Reproduces original Symile unique-admission batching, retaining the existing relative-L2 and cosine acceptance thresholds. Run in the original RAD-DINO CUDA environment. This notebook changes no frozen Python source, model weights or stored embeddings, so the current experiment protocol remains usable. It preserves prior audits before rechecking. A failed recheck still blocks bank finalization.


In [ ]:
from pathlib import Path
import sys
EXP = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / 'config.example.json').is_file() and (p / 'common.py').is_file()), None)
if EXP is None: raise FileNotFoundError('Start Jupyter inside graph_reasoning/experiment02')
sys.path.insert(0, str(EXP))
from common import load_config, parent_setup, read
CONFIG_PATH = EXP / 'config.json'
if not CONFIG_PATH.is_file():
    raise FileNotFoundError('Run BR00 first to create experiment02/config.json')
CFG = load_config(CONFIG_PATH)
PROJECT = parent_setup(CFG)
RUN = Path(CFG['run_dir'])
print('Kernel:', sys.executable)
print('Cohort:', CFG['cohort'], '| outputs:', RUN)


In [ ]:
import hashlib
from unittest.mock import patch
import numpy as np
from PIL import Image
import encoders
from common import sha, atomic, immutable

SNAPSHOT_PATH = None  # Same original microsoft/rad-dino snapshot.
PROBES = 5
ORIGINAL_BATCH_SIZE = 16  # Default in notebook2/04_encode_cxr_raddino.ipynb.
# If the ORIGINAL encoding job overrode RADDINO_BATCH_SIZE, use that exact value.
if type(ORIGINAL_BATCH_SIZE) is not int or ORIGINAL_BATCH_SIZE < 1:
    raise ValueError('Original batch size must be a positive integer')

bank = read(RUN / 'banks/symile.json')
folder = Path(CFG['symile_root']) / 'data_npy/train'
raw = np.load(folder / 'cxr_train.npy', mmap_mode='r', allow_pickle=False)
hadm = np.load(folder / 'hadm_id_train.npy', mmap_mode='r', allow_pickle=False).reshape(-1)
if len(raw) != len(hadm): raise ValueError('CXR/admission array length mismatch')
# Original encoder processed UNIQUE admissions in np.unique's sorted order.
_, first_index, inverse = np.unique(hadm, return_index=True, return_inverse=True)
rng = np.random.default_rng(CFG['seed'])
probe_rows = [bank[i] for i in rng.choice(len(bank), min(PROBES, len(bank)), replace=False)]

def to_image(array):
    x = np.asarray(array, dtype=np.float32)
    x = x * np.array([.229, .224, .225], dtype=np.float32)[:, None, None]
    x = x + np.array([.485, .456, .406], dtype=np.float32)[:, None, None]
    return Image.fromarray((np.clip(x, 0, 1).transpose(1, 2, 0) * 255).round().astype('uint8'))

def pixel_hash(image):
    x = np.ascontiguousarray(np.asarray(image.convert('RGB')))
    return hashlib.sha256(x.tobytes()).hexdigest()

original_encode = encoders.rad_encode
state = {'calls': 0, 'batches': {}, 'details': []}

def encode_with_original_batches(bundle, image):
    number = state['calls']
    state['calls'] += 1
    # The verifier processes all Symile probes first, then MIMIC probes.
    # MIMIC already passed single-image checks and retains the same procedure.
    if number >= len(probe_rows):
        return original_encode(bundle, image)
    row = probe_rows[number]
    source_index = int(row['source_index'])
    if str(hadm[source_index]).removesuffix('.0') != str(row['admission_id']):
        raise ValueError('Probe admission mismatch')
    if pixel_hash(image) != pixel_hash(to_image(raw[source_index])):
        raise ValueError('Unexpected verifier probe order or image preprocessing')
    unique_position = int(inverse[source_index])
    # Original pipeline encoded position 0 alone as its first probe, then 1..N in batches.
    start = 0 if unique_position == 0 else 1 + ((unique_position - 1) // ORIGINAL_BATCH_SIZE) * ORIGINAL_BATCH_SIZE
    stop = 1 if start == 0 else min(start + ORIGINAL_BATCH_SIZE, len(first_index))
    key = (start, stop)
    if key not in state['batches']:
        images = [to_image(raw[int(i)]) for i in first_index[start:stop]]
        torch = bundle['torch']
        with torch.inference_mode():
            pixels = bundle['processor'](images=images, return_tensors='pt')['pixel_values']
            output = bundle['model'](pixel_values=pixels.to('cuda', dtype=torch.float16))
            pooled = output.pooler_output if getattr(output, 'pooler_output', None) is not None else output.last_hidden_state[:, 0]
        state['batches'][key] = pooled.float().cpu().numpy()
    state['details'].append({'id': row['id'], 'source_index': source_index,
                             'unique_admission_position': unique_position,
                             'batch_start': start, 'batch_stop': stop})
    return state['batches'][key][unique_position - start]


In [ ]:
audit_path = RUN / 'audits/rad_equivalence.json'
if audit_path.exists():
    previous = read(audit_path)
    # Retain every prior audit by content hash; never erase the failed verification history.
    from common import signature
    immutable(RUN / 'audits/rad_history' / (signature(previous) + '.json'), previous)

previous_stamp = audit_path.stat().st_mtime_ns if audit_path.exists() else None
error = None
try:
    with patch.object(encoders, 'rad_encode', side_effect=encode_with_original_batches):
        result = encoders.verify_rad(CFG, snapshot=SNAPSHOT_PATH, probes=PROBES)
except ValueError as exc:
    error = exc
finally:
    if audit_path.exists() and audit_path.stat().st_mtime_ns != previous_stamp:
        result = read(audit_path)
        result['verification_procedure'] = {
            'name': 'original_unique_admission_batch_reproduction',
            'original_batch_size': ORIGINAL_BATCH_SIZE,
            'symile_probe_batches': state['details'],
            'mimic_procedure': 'unchanged single-image verification',
            'thresholds_unchanged': {'max_relative_l2': 0.005, 'min_cosine': 0.99999},
            'recheck_notebook_sha256': sha(EXP / 'BR01A_batch_matched_rad_recheck.ipynb'),
            'note': 'No embedding, checkpoint, threshold or frozen Python helper was changed.'
        }
        atomic(audit_path, result)
        for check in result.get('checks', []): print(check)
        print('Verification passed:', result['passed'])
        print('Audit:', audit_path)

if error is not None:
    raise ValueError('Batch-matched recheck still failed. Keep the gate closed and inspect the updated audit; do not lower thresholds.') from error
